# YouTube JSONL Preprocessing
Removes label R, keeps only high-confidence rows, exports JSONL and CSV, removes Confidence and __row_index, and verifies outputs.

In [ ]:
import json
import pandas as pd
from pathlib import Path

file_path='youtube_labeled_full.jsonl'

with open(file_path,'r',encoding='utf-8') as f:
    first_row=json.loads(f.readline())

print('Columns:', list(first_row.keys()))

Columns: ['index', 'Label', 'Confidence', 'Reasoning', '__row_index', 'id', 'video id', 'author', 'text', 'likeCount', 'created_time', 'video_date']


In [ ]:
# FILTER DATASET
# 1) Remove Label == 'R'
# 2) Keep ONLY High confidence rows
# 3) Export JSONL + CSV

import json
import pandas as pd
from pathlib import Path

input_file=Path('youtube_labeled_full.jsonl')
output_jsonl=Path('youtube_filtered_high_confidence.jsonl')
output_csv=Path('youtube_filtered_high_confidence.csv')

with input_file.open('r',encoding='utf-8') as f:
    first_row=json.loads(f.readline())

label_key='Label' if 'Label' in first_row else 'label'
conf_key='Confidence' if 'Confidence' in first_row else 'confidence'

HIGH_CONF_VALUES={'high','very high','high confidence','very_high','strong'}

kept=0
removed_r=0
removed_low_conf=0
rows=[]

with input_file.open('r',encoding='utf-8') as fin, output_jsonl.open('w',encoding='utf-8') as fout:
    for line in fin:
        row=json.loads(line)

        label=str(row.get(label_key,'')).strip().upper()
        if label=='R':
            removed_r+=1
            continue

        conf=str(row.get(conf_key,'')).strip().lower()
        if conf not in HIGH_CONF_VALUES:
            removed_low_conf+=1
            continue

        fout.write(json.dumps(row,ensure_ascii=False)+'\n')
        rows.append(row)
        kept+=1

pd.DataFrame(rows).to_csv(output_csv,index=False)

print('Kept:',kept)
print('Removed R:',removed_r)
print('Removed Low Confidence:',removed_low_conf)

Kept: 378293
Removed R: 116813
Removed Low Confidence: 245972


In [ ]:
# REMOVE CONFIDENCE FROM JSONL
import json
from pathlib import Path

input_file=Path('youtube_filtered_high_confidence.jsonl')
output_file=Path('youtube_filtered_final.jsonl')

with input_file.open('r',encoding='utf-8') as fin, output_file.open('w',encoding='utf-8') as fout:
    for line in fin:
        row=json.loads(line)
        row.pop('Confidence',None)
        row.pop('confidence',None)
        fout.write(json.dumps(row,ensure_ascii=False)+'\n')

print('Saved',output_file)

Saved youtube_filtered_final.jsonl


In [ ]:
# REMOVE CONFIDENCE FROM CSV
import pandas as pd

df=pd.read_csv('youtube_filtered_high_confidence.csv')
df=df.drop(columns=[c for c in ['Confidence','confidence'] if c in df.columns])
df.to_csv('youtube_filtered_final.csv',index=False)
print(df.shape)

(378293, 11)


In [ ]:
# REMOVE __row_index
import pandas as pd

df=pd.read_csv('youtube_filtered_final.csv')
if '__row_index' in df.columns:
    df=df.drop(columns=['__row_index'])

df.to_csv('youtube_dataset_final.csv',index=False)
print('Final shape:',df.shape)
print(df.columns.tolist())

Final shape: (378293, 10)
['index', 'Label', 'Reasoning', 'id', 'video id', 'author', 'text', 'likeCount', 'created_time', 'video_date']
